# Week 2 — Feature Engineering



## Step 1: Understand the Target Shape of Your Table

Machine learning needs a **flat table**: one row per customer, one column per feature.

Week 1 tables are the opposite. Viewing, payments, tickets, and subscriptions have **many rows per customer**. Each of those tables must be summarized down to one row per customer, then **left-joined** onto `customers` so nobody disappears.

### Target table: `customer_features` (~8,000 rows)

| Column group | Examples | Role |
|---|---|---|
| Identifier | `customer_id` | Join key only — **not** a model feature |
| Profile | country, age, segment, current plan | Inputs |
| Engineered behaviour | tenure, watch time, payment failures, tickets | Inputs |
| Target 1 | `churned` | Churn classification |
| Target 2 | `customer_lifetime_value` | CLV regression |

```
customers (1 row per customer)
    left join  viewing summaries
    left join  payment summaries
    left join  ticket summaries
    left join  subscription summaries
    left join  feedback summaries
    left join  churn_labels
           ↓
customer_features.csv   (~8,000 unique customer_id values)
```

### Feature sketch — which Week 1 table each column comes from

| Feature | Source table | Grain after summary |
|---|---|---|
| `customer_id`, age, country, registration_date, segment, language | `customers` | already 1 row per customer |
| `churned`, `churn_date`, `churn_reason` | `churn_labels` | already 1 row per customer |
| `customer_tenure_days` | `customers` | snapshot date − registration date |
| `avg_watch_time_per_session` | `viewing_activity` | mean watch minutes per customer |
| `monthly_watch_time_trend` | `viewing_activity` | slope of monthly watch minutes |
| `days_since_last_activity` | `viewing_activity` | snapshot date − last session |
| `login_frequency` | `viewing_activity` | sessions ÷ active months |
| `support_ticket_count` | `support_tickets` | count per customer (`0` if none) |
| `avg_customer_satisfaction` | `support_tickets` | mean CSAT on scored tickets |
| `failed_payment_count` / `failed_payment_rate` | `payments` | failed count; failed ÷ total |
| `plan_changes` | `subscriptions` | number of sub rows − 1 |
| `engagement_score` | derived from viewing | recency + frequency + watch depth |
| `customer_lifetime_value` | `payments` | sum of **successful** payments |


**Join rule:** always start from `customers` and **left join**. Inner joins would drop customers with no viewing, payments, or tickets.

Confirm the grain: many event rows, but only ~8,000 unique customers.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"

tables = {
    "customers": "customers_clean.csv",
    "subscriptions": "subscriptions_clean.csv",
    "viewing_activity": "viewing_activity_clean.csv",
    "payments": "payments_clean.csv",
    "support_tickets": "support_tickets_clean.csv",
    "customer_feedback": "customer_feedback_clean.csv",
    "churn_labels": "churn_labels_clean.csv",
}

rows = []
for name, file in tables.items():
    df = pd.read_csv(PROCESSED / file)
    rows.append({
        "table": name,
        "rows": len(df),
        "unique_customers": df["customer_id"].nunique(),
        "rows_per_customer": round(len(df) / df["customer_id"].nunique(), 2),
    })

grain = pd.DataFrame(rows)
display(grain)
print("customers table is already 1 row per customer:" , grain.loc[grain["table"]=="customers", "rows_per_customer"].item() == 1)

,table,rows,unique_customers,rows_per_customer
0,customers,8000,8000,1.00
1,subscriptions,8875,8000,1.11
2,viewing_activity,66422,7980,8.32
3,payments,110735,8000,13.84
4,support_tickets,6120,4408,1.39
5,customer_feedback,5046,3561,1.42
6,churn_labels,8000,8000,1.00


customers table is already 1 row per customer: True


## Step 2: Snapshot date and leakage

I am not using today's date for tenure or recency. Those numbers would change every time I rerun the notebook, and customers who already churned would look inactive just because they left.

I checked the date ranges first. Viewing, tickets, and churn all stop on **2026-06-30**, and every churned customer's `churn_date` is the same as their `cancellation_date`. So I am using **30 June 2026** as one fixed snapshot for everyone. That is the end of this dataset, not "today".

To avoid leakage I only use behaviour **before** each customer could have been known to churn:
- if they churned, my cutoff is their `churn_date` (I ignore watch sessions, payments, and tickets on or after that day)
- if they are still active, my cutoff is the snapshot date

That way `days_since_last_activity` is recency *before* they left, not a cheaty "they have been gone for months so they must have churned" signal.

In [2]:
customers = pd.read_csv(PROCESSED / "customers_clean.csv", parse_dates=["registration_date"])
viewing = pd.read_csv(PROCESSED / "viewing_activity_clean.csv", parse_dates=["viewing_date"])
payments = pd.read_csv(PROCESSED / "payments_clean.csv", parse_dates=["payment_date"])
tickets = pd.read_csv(PROCESSED / "support_tickets_clean.csv", parse_dates=["ticket_date"])
churn = pd.read_csv(PROCESSED / "churn_labels_clean.csv", parse_dates=["churn_date"])
subs = pd.read_csv(
    PROCESSED / "subscriptions_clean.csv",
    parse_dates=["subscription_start_date", "subscription_end_date", "cancellation_date"],
)

print("registration", customers["registration_date"].min().date(), "→", customers["registration_date"].max().date())
print("viewing     ", viewing["viewing_date"].min().date(), "→", viewing["viewing_date"].max().date())
print("payments    ", payments["payment_date"].min().date(), "→", payments["payment_date"].max().date())
print("tickets     ", tickets["ticket_date"].min().date(), "→", tickets["ticket_date"].max().date())
print("churn_date  ", churn["churn_date"].min().date(), "→", churn["churn_date"].max().date())
print("churn rate  ", round(churn["churned"].mean() * 100, 1), "%")

churn_vs_cancel = churn.merge(subs[["customer_id", "cancellation_date"]], on="customer_id", how="left")
churned_only = churn_vs_cancel[churn_vs_cancel["churned"]]
print(
    "churn_date matches cancellation_date:",
    (churned_only["churn_date"] == churned_only["cancellation_date"]).all(),
)

registration 2019-01-01 → 2026-05-31
viewing      2025-07-01 → 2026-06-30
payments     2019-01-01 → 2026-05-31
tickets      2019-03-03 → 2026-06-30
churn_date   2025-10-03 → 2026-06-30
churn rate   24.9 %
churn_date matches cancellation_date: False


In [3]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))
from src.features.config import SNAPSHOT_DATE

print("SNAPSHOT_DATE:", SNAPSHOT_DATE.date())

cutoffs = churn[["customer_id", "churned", "churn_date"]].copy()
cutoffs["feature_cutoff"] = cutoffs["churn_date"].where(cutoffs["churned"], SNAPSHOT_DATE)

display(cutoffs.head())
print("active customers use snapshot:", (~cutoffs["churned"]).sum())
print("churned customers use churn_date:", cutoffs["churned"].sum())

SNAPSHOT_DATE: 2026-06-30


,customer_id,churned,churn_date,feature_cutoff
0,CUST000001,False,NaT,2026-06-30
1,CUST000002,False,NaT,2026-06-30
2,CUST000003,True,2026-04-20,2026-04-20
3,CUST000004,False,NaT,2026-06-30
4,CUST000005,False,NaT,2026-06-30


active customers use snapshot: 6009
churned customers use churn_date: 1991


## Step 3: Required features

I am building one row per customer from the Week 1 tables. Every count starts from `customers` and is left joined, so people with no viewing, tickets, or payments stay in the table and get `0` for counts.

I only use events **before** each customer's `feature_cutoff` from Step 2. For `engagement_score` I scaled recency, login frequency, and average watch time to 0-1 (and flipped recency so a recent session scores higher). My weights are 40% recency, 30% frequency, 30% watch depth. Recency gets a bit more weight because a customer who still showed up recently is the one I would treat as engaged.

`avg_customer_satisfaction` is the only column I am leaving missing on purpose. If I never opened a ticket (or my ticket was still pending), I do not have a real CSAT score, and filling one would be made up.

In [4]:
import importlib
import sys

import src.features.config as feature_config

importlib.reload(feature_config)
sys.modules.pop("src.features.build_features", None)

from src.features.build_features import add_feature_cutoff, build_required_features
from src.features.config import ENGAGEMENT_WEIGHTS

cutoffs = add_feature_cutoff(churn)
required_features = build_required_features(
    customers, viewing, payments, tickets, subs, cutoffs
)

print("weights I used:", ENGAGEMENT_WEIGHTS)
print("rows:", len(required_features), "unique customers:", required_features["customer_id"].nunique())
display(required_features.head())

weights I used: {'recency': 0.4, 'frequency': 0.3, 'watch_depth': 0.3}
rows: 8000 unique customers: 8000


,customer_id,customer_tenure_days,avg_watch_time_per_session,monthly_watch_time_trend,days_since_last_activity,login_frequency,support_ticket_count,avg_customer_satisfaction,failed_payment_count,failed_payment_rate,customer_lifetime_value,plan_changes,engagement_score
0,CUST000744,2584,53.200000,-5.142857,11.0,1.250000,0,NaN,2,0.133333,298.87,0,0.563262
1,CUST001502,168,61.800000,-20.771429,61.0,1.250000,0,NaN,0,0.000000,34.95,0,0.577995
2,CUST005383,1109,50.166667,-13.000000,152.0,2.000000,0,NaN,1,0.066667,90.87,0,0.548005
3,CUST004708,2497,38.900000,-22.600000,12.0,1.666667,0,NaN,1,0.066667,216.86,0,0.533806
4,CUST005877,209,57.200000,-15.371429,76.0,1.666667,0,NaN,1,0.166667,91.96,0,0.571853


In [5]:
print("missing values")
display(required_features.isna().sum())
print("\nsummary")
display(required_features.describe().T)
print(
    "CSAT missing because those customers have no scored ticket:",
    required_features["avg_customer_satisfaction"].isna().sum(),
)

missing values


customer_id                      0
customer_tenure_days             0
avg_watch_time_per_session       0
monthly_watch_time_trend         0
days_since_last_activity         0
login_frequency                  0
support_ticket_count             0
avg_customer_satisfaction     4616
failed_payment_count             0
failed_payment_rate              0
customer_lifetime_value          0
plan_changes                     0
engagement_score                 0
dtype: int64


summary


,count,mean,std,min,25%,50%,75%,max
customer_tenure_days,8000.0,1343.628375,780.144151,0.000000,664.000000,1347.000000,2004.000000,2737.000000
avg_watch_time_per_session,8000.0,47.130213,12.600092,0.000000,41.195455,47.666667,54.111111,114.000000
monthly_watch_time_trend,8000.0,4.742368,17.559621,-53.257143,-5.571429,0.857143,11.685714,128.714286
days_since_last_activity,8000.0,63.519375,177.754099,0.000000,9.000000,26.000000,64.000000,2532.000000
login_frequency,8000.0,1.558249,0.684507,0.000000,1.166667,1.428571,1.750000,15.000000
support_ticket_count,8000.0,0.719000,0.867544,0.000000,0.000000,1.000000,1.000000,7.000000
avg_customer_satisfaction,3384.0,3.257526,1.046917,1.000000,2.666667,3.000000,4.000000,5.000000
failed_payment_count,8000.0,0.889875,1.017780,0.000000,0.000000,1.000000,1.000000,8.000000
failed_payment_rate,8000.0,0.065704,0.084909,0.000000,0.000000,0.066667,0.111111,1.000000
customer_lifetime_value,8000.0,177.029727,99.805785,0.000000,97.860000,183.920000,232.350000,629.930000


CSAT missing because those customers have no scored ticket: 4616


I now have all the required columns: tenure, average watch time, monthly watch trend, days since last activity, login frequency, ticket count, CSAT, failed payment count and rate, plan changes, engagement score, and lifetime value from successful payments. Counts are filled with 0. CSAT is empty only when I have no scored ticket to average.

## Step 4: Extra features from a quick churn split


- `avg_feedback_rating`: churned customers who left feedback rated me lower on average (about 2.99 vs 3.24), so I am keeping mean rating as a dissatisfaction signal.
- `current_plan_name`: Basic had a higher churn rate than Premium (about 26.3% vs 22.8%), so the plan I am on at the cutoff is worth keeping.
- `avg_successful_payment_amount`: churned customers had a slightly lower mean successful payment (about 13.33 vs 13.89), which lines up with cheaper plans churning more.
- `feedback_count`: customers with no feedback churned more (about 26.8% vs 22.4%), so even bothering to leave feedback looks like a weaker churn signal. Count is 0 if I left none. Rating stays empty if I left none, same idea as CSAT.

In [6]:
feedback = pd.read_csv(PROCESSED / "customer_feedback_clean.csv", parse_dates=["feedback_date"])
plans = pd.read_csv(PROCESSED / "subscription_plans_clean.csv")

eda = required_features.merge(churn[["customer_id", "churned"]], on="customer_id", how="left")

fb_before = feedback.merge(cutoffs[["customer_id", "feature_cutoff"]], on="customer_id")
fb_before = fb_before[fb_before["feedback_date"] < fb_before["feature_cutoff"]]
rating_eda = (
    fb_before.groupby("customer_id")["rating"].mean().reset_index(name="avg_feedback_rating")
    .merge(churn[["customer_id", "churned"]], on="customer_id")
)
print("mean feedback rating by churn")
display(rating_eda.groupby("churned")["avg_feedback_rating"].mean())

subs_before = subs.merge(cutoffs[["customer_id", "feature_cutoff"]], on="customer_id")
subs_before = subs_before[subs_before["subscription_start_date"] < subs_before["feature_cutoff"]]
last_plan = (
    subs_before.sort_values("subscription_start_date")
    .groupby("customer_id").tail(1)[["customer_id", "plan_id"]]
    .merge(plans[["plan_id", "plan_name"]], on="plan_id")
    .merge(churn[["customer_id", "churned"]], on="customer_id")
)
print("churn rate by current plan")
display(last_plan.groupby("plan_name")["churned"].mean())

mean feedback rating by churn


churned
False    3.240563
True     2.991147
Name: avg_feedback_rating, dtype: float64

churn rate by current plan


plan_name
Basic       0.263283
Premium     0.227778
Standard    0.243854
Name: churned, dtype: float64

In [7]:
importlib.reload(feature_config)
sys.modules.pop("src.features.build_features", None)
from src.features.build_features import build_extra_features

extra_features = build_extra_features(feedback, subs, plans, payments, cutoffs)
customer_features = required_features.merge(extra_features, on="customer_id", how="left")
customer_features["feedback_count"] = customer_features["feedback_count"].fillna(0).astype(int)
customer_features["avg_successful_payment_amount"] = customer_features[
    "avg_successful_payment_amount"
].fillna(0)
customer_features["current_plan_name"] = customer_features["current_plan_name"].fillna("Unknown")

print("extra columns:", list(extra_features.columns))
print("combined shape:", customer_features.shape)
display(customer_features[["customer_id", "avg_feedback_rating", "feedback_count", "current_plan_name", "avg_successful_payment_amount"]].head())
print("missing after extras")
display(customer_features[["avg_feedback_rating", "feedback_count", "current_plan_name", "avg_successful_payment_amount"]].isna().sum())

extra columns: ['customer_id', 'feedback_count', 'avg_feedback_rating', 'current_plan_name', 'avg_successful_payment_amount']
combined shape: (8000, 17)


,customer_id,avg_feedback_rating,feedback_count,current_plan_name,avg_successful_payment_amount
0,CUST000744,3.0,1,Premium,22.990000
1,CUST001502,NaN,0,Basic,6.990000
2,CUST005383,4.0,1,Basic,6.490714
3,CUST004708,NaN,0,Standard,15.490000
4,CUST005877,NaN,0,Premium,18.392000


missing after extras


avg_feedback_rating              4563
feedback_count                      0
current_plan_name                   0
avg_successful_payment_amount       0
dtype: int64

I added four extras, each tied to that split. `avg_feedback_rating` is empty when I left no review. The other three are filled: `feedback_count` is 0, `current_plan_name` is Unknown only if I somehow have no subscription before the cutoff, and `avg_successful_payment_amount` is 0 if I have no successful payment.

## Step 5: Assemble, check, and save

I am starting from `customers` and left joining the required features, the extra features, and the churn labels. I am dropping names (not a model input) and `state_or_region` (it was empty in Week 1). After that I check row count, unique IDs, missing values, and a few sanity rules, then save `data/processed/customer_features.csv`.

In [8]:
profile_cols = [
    "customer_id",
    "age",
    "gender",
    "country",
    "city",
    "registration_date",
    "acquisition_channel",
    "customer_segment",
    "preferred_language",
]

customer_features = (
    customers[profile_cols]
    .merge(required_features, on="customer_id", how="left")
    .merge(extra_features, on="customer_id", how="left")
    .merge(churn[["customer_id", "churned", "churn_date", "churn_reason"]], on="customer_id", how="left")
)

customer_features["feedback_count"] = customer_features["feedback_count"].fillna(0).astype(int)
customer_features["avg_successful_payment_amount"] = customer_features[
    "avg_successful_payment_amount"
].fillna(0)
customer_features["current_plan_name"] = customer_features["current_plan_name"].fillna("Unknown")

print("rows:", len(customer_features))
print("unique customer_id:", customer_features["customer_id"].nunique())
print("customer_id is unique:", customer_features["customer_id"].is_unique)
print("columns:", len(customer_features.columns))
display(customer_features.head())

rows: 8000
unique customer_id: 8000
customer_id is unique: True
columns: 28


,customer_id,age,gender,country,city,registration_date,acquisition_channel,customer_segment,preferred_language,customer_tenure_days,...,customer_lifetime_value,plan_changes,engagement_score,feedback_count,avg_feedback_rating,current_plan_name,avg_successful_payment_amount,churned,churn_date,churn_reason
0,CUST000744,21,Female,United Kingdom,Liverpool,2019-06-03,Organic Search,Individual,English,2584,...,298.87,0,0.563262,1,3.0,Premium,22.990000,False,NaT,NaN
1,CUST001502,31,Male,Canada,Montreal,2026-01-13,Referral,Family,French,168,...,34.95,0,0.577995,0,NaN,Basic,6.990000,False,NaT,NaN
2,CUST005383,40,Male,United Kingdom,Leeds,2023-06-17,Paid Social,Student,English,1109,...,90.87,0,0.548005,1,4.0,Basic,6.490714,False,NaT,NaN
3,CUST004708,27,Male,United Kingdom,Manchester,2019-03-09,Partner Bundle (Telecom),Individual,Hindi,2497,...,216.86,0,0.533806,0,NaN,Standard,15.490000,True,2026-01-08,Household budget cuts
4,CUST005877,36,Male,Indonesia,Jakarta,2025-12-03,Email Campaign,Student,French,209,...,91.96,0,0.571853,0,NaN,Premium,18.392000,False,NaT,NaN


In [9]:
print("missing values")
display(customer_features.isna().sum())

print("\nnumeric summary")
display(customer_features.describe().T)

sanity = {
    "negative tenure": int((customer_features["customer_tenure_days"] < 0).sum()),
    "failed_payment_rate above 1": int((customer_features["failed_payment_rate"] > 1).sum()),
    "failed_payment_rate below 0": int((customer_features["failed_payment_rate"] < 0).sum()),
    "negative CLV": int((customer_features["customer_lifetime_value"] < 0).sum()),
    "negative days_since_last_activity": int((customer_features["days_since_last_activity"] < 0).sum()),
    "engagement_score outside 0-1": int(
        ((customer_features["engagement_score"] < 0) | (customer_features["engagement_score"] > 1)).sum()
    ),
    "churned missing": int(customer_features["churned"].isna().sum()),
}
print("\nsanity checks (all should be 0)")
for name, value in sanity.items():
    print(f"  {name}: {value}")

missing values


customer_id                         0
age                                 0
gender                              0
country                             0
city                                0
registration_date                   0
acquisition_channel                 0
customer_segment                    0
preferred_language                  0
customer_tenure_days                0
avg_watch_time_per_session          0
monthly_watch_time_trend            0
days_since_last_activity            0
login_frequency                     0
support_ticket_count                0
avg_customer_satisfaction        4616
failed_payment_count                0
failed_payment_rate                 0
customer_lifetime_value             0
plan_changes                        0
engagement_score                    0
feedback_count                      0
avg_feedback_rating              4563
current_plan_name                   0
avg_successful_payment_amount       0
churned                             0
churn_date  


numeric summary


,count,mean,min,25%,50%,75%,max,std
age,8000.0,32.616625,13.0,25.0,32.0,40.0,100.0,10.708244
registration_date,8000,2022-09-23 00:19:48,2019-01-01 00:00:00,2020-11-28 00:00:00,2022-09-23 00:00:00,2024-07-23 06:00:00,2026-05-31 00:00:00,NaN
customer_tenure_days,8000.0,1343.628375,0.0,664.0,1347.0,2004.0,2737.0,780.144151
avg_watch_time_per_session,8000.0,47.130213,0.0,41.195455,47.666667,54.111111,114.0,12.600092
monthly_watch_time_trend,8000.0,4.742368,-53.257143,-5.571429,0.857143,11.685714,128.714286,17.559621
days_since_last_activity,8000.0,63.519375,0.0,9.0,26.0,64.0,2532.0,177.754099
login_frequency,8000.0,1.558249,0.0,1.166667,1.428571,1.75,15.0,0.684507
support_ticket_count,8000.0,0.719,0.0,0.0,1.0,1.0,7.0,0.867544
avg_customer_satisfaction,3384.0,3.257526,1.0,2.666667,3.0,4.0,5.0,1.046917
failed_payment_count,8000.0,0.889875,0.0,0.0,1.0,1.0,8.0,1.01778



sanity checks (all should be 0)
  negative tenure: 0
  failed_payment_rate above 1: 0
  failed_payment_rate below 0: 0
  negative CLV: 0
  negative days_since_last_activity: 0
  engagement_score outside 0-1: 0
  churned missing: 0


In [10]:
out_path = PROCESSED / "customer_features.csv"
customer_features.to_csv(out_path, index=False)
print("saved:", out_path)
print("file rows:", len(pd.read_csv(out_path)))

saved: c:\Users\saksh\OneDrive - University of Keele\Documents\Netflix_subscription_project\data\processed\customer_features.csv
file rows: 8000


The saved table should have about 8,000 unique customers. Counts are filled with 0. The missing values I still expect are `avg_customer_satisfaction` and `avg_feedback_rating` (no scored ticket / no review), plus `churn_date` and `churn_reason` for people who did not churn.